# Utils y librerías




In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# UTILS

import pandas as pd
import unicodedata
import glob
import os
import matplotlib.pyplot as plt

from openai import OpenAI
from tqdm import tqdm
import numpy as np

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

import nltk
nltk.download('stopwords')
nltk.download('punkt') # for word_tokenize
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# 1.Archivos de caracterización



Edunext es la primera plataforma, utilizada desde 2022 a 2023.
Openedx, por su parte, se usó de 2023 a 2025.

Se cuenta con tres archivos de caracterización.
Dos corresponden a openedx y el tercero a edunext.

Primero, se quiere saber si los dos archivos correspondientes a openedx contienen la misma información o información duplicada, por lo que se examinarán sus columnas y se unificarán si es necesario.

##1.1 OPENEDX

In [ ]:
openedx_v1_char_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/20260212/characterization/characterization_openedx.csv'
openedx_v2_char_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/20260212/characterization/characterization_openedx_v2.csv'


In [ ]:
openedx_char_v1 = pd.read_csv(openedx_v1_char_path)
openedx_char_v2 = pd.read_csv(openedx_v2_char_path)


In [ ]:
print("Columnas de archivo 1 de caracterización", openedx_char_v1.columns)

Las columnas de cada uno de los archivos de caracterización son:

v1:

*   Unnamed
*   username
*   Encuesta inicial
*   Motivación
*   Ocupacion,
*   lenguajes_programacion
*   nivel programación
*   Dedicación semanal
*   razon_realizar_curso
*   otro_medio
*   otra_motivacion
*   otro_lenguaje
*   otra_ocupacion
*   email
*   name
*   uncode_usr

In [ ]:
print("Columnas de archivo 1 de caracterización", openedx_char_v2.columns)

v2:


*   Unnamed
*   username
*   Donde conoció el curso
*   Encuesta inicial
*   Motivación
*   Ocupacion,
*   lenguajes_programacion
*   nivel programación
*   Dedicación semanal
*   razon_realizar_curso
*   otro_medio
*   otra_motivacion
*   otro_lenguaje
*   otra_ocupacion
*   email
*   name
*   uncode_usr

El archivo v2 posee una columna más que el v1, relacionada a dónde conoció el curso. Por esto, se agregará la columna al primer archivo.

In [ ]:
openedx_char_v1.insert(2, 'Donde conoció el curso', 'NA')

In [ ]:
print("Columnas de archivo 1 de caracterización", openedx_char_v1.columns)

In [ ]:
print("Columnas de archivo 2 de caracterización", openedx_char_v2.columns)

Limpieza de nombres de columnas:

In [ ]:
openedx_char_v1.columns = openedx_char_v1.columns.str.lower().str.replace(' ', '')
openedx_char_v2.columns = openedx_char_v2.columns.str.lower().str.replace(' ', '')
print("Nuevas columnas", openedx_char_v1.columns)
print("Nuevas columnas", openedx_char_v2.columns)

Se eliminan tildes y caracteres especiales de los nombres de las columnas

In [ ]:
# removing spanish characters

def remove_accents(text):
    return (
        unicodedata
        .normalize('NFKD', text)
        .encode('ascii', 'ignore')
        .decode('utf-8')
    )

openedx_char_v1.columns = [remove_accents(col) for col in openedx_char_v1.columns]
openedx_char_v2.columns = [remove_accents(col) for col in openedx_char_v2.columns]

print(openedx_char_v1.columns)
print(openedx_char_v2.columns)

Se agrega una columna final donde se especifica que el registro proviene de openedx.

In [ ]:
openedx_char_v1['platform'] = 'openedx'
openedx_char_v2['platform'] = 'openedx'

Se elimina la columna unnamed ya que puede causar confusión a la hora de querer comparar ambos dataframes.

In [ ]:
openedx_char_v1.drop(columns=['unnamed:0'], inplace=True)
openedx_char_v2.drop(columns=['unnamed:0'], inplace=True)

El primer archivo de caracterización tiene 826 registros; el segundo 525

In [ ]:
len(openedx_char_v1)

In [ ]:
len(openedx_char_v2)

Se realiza una limpieza de la columna de email pues se utilizará como identificador para comparar los registros

In [ ]:
openedx_char_v2['email'] = openedx_char_v2['email'].str.strip().str.lower()
openedx_char_v1['email'] = openedx_char_v1['email'].str.strip().str.lower()


In [ ]:
comparacion = openedx_char_v2[['email']].merge(
    openedx_char_v1[['email']],
    on='email',
    how='left',
    indicator=True
)

todo_contenido = (comparacion['_merge'] == 'both').all()
emails_faltantes = comparacion[comparacion['_merge'] == 'left_only']['email']

print(f"¿Están todos los emails?: {todo_contenido}")

Se realiza una comparación entre el archivo v1 y v2, para ver si uno contiene al otro. Por medio de un merge (left join) se observa que de 525 emails en el archivo v2, 355 no están presentes en la v1.

In [ ]:
emails_faltantes

se realiza una búsqueda en forma de verificación usando los emails listados anteriormente, para comprobar que están presentes en la v2, pero no en la v1

In [ ]:
openedx_char_v1[openedx_char_v1['email'] == '@gmail.com'] # modificada por proteccion de datos

In [ ]:
openedx_char_v2[openedx_char_v2['email'] == '@gmail.com'] # modificada por proteccion de datos

De esta manera, no se pueden descartar ninguno de los dos archivos pues es una gran cantidad de registros del archivo "pequeño" los que no están contenidos en el segundo archivo.

Así, se procede a juntar ambos en uno solo.
Al juntarlo, ahora tenemos 1351 filas

In [ ]:
openedx_char = pd.concat([openedx_char_v1, openedx_char_v2], ignore_index=True)

In [ ]:
len(openedx_char)

In [ ]:
openedx_char.columns

Se observa, entonces, los registros duplicados, de nuevo, por email.

In [ ]:
duplicados = openedx_char[openedx_char.duplicated(subset=['email'], keep=False)]

# ordenar por email para que los repetidos aparezcan juntos
duplicados_ordenados = duplicados.sort_values(by='email')

In [ ]:
duplicados_ordenados

Se puede observar que los registros duplicados son iguales, exceptuando por el valor de la columna 'donde conoció el curso'. Esto significa que el archivo v2 tiene información de dónde el estudiante conoció el curso que no estaba disponible para algunos registros en la v1.




In [ ]:
openedx_char_v1[openedx_char_v1['email'] == '@gmail.com'] # modificada por proteccion de datos

In [ ]:
openedx_char_v2[openedx_char_v2['email'] == '@gmail.com'] # modificada por proteccion de datos

In [ ]:
openedx_char[openedx_char['email'] == '@gmail.com'] # modificada por proteccion de datos

Por esto, se realizará la eliminación de los email duplicados, solamente para aquellos registros que no tengan información de dónde se conoció el curso, para asi no perderla.

Se deberian eliminar 170 registros, para un total de final de 1181 registros de caracterización únicos.


In [ ]:
openedx_char['es_valido'] = openedx_char['dondeconocioelcurso'] != 'NA'
final_openedx_char = openedx_char.sort_values(by=['email', 'es_valido'], ascending=[True, False])
final_openedx_char = final_openedx_char.drop_duplicates(subset=['email'], keep='first')
final_openedx_char = final_openedx_char.drop(columns=['es_valido'])

In [ ]:
len(final_openedx_char)

In [ ]:
final_openedx_char.columns

Verificamos que el duplicado haya sido eliminado y que nos quedamos con el registro que cuenta con información

In [ ]:
final_openedx_char[final_openedx_char['email'] == '@gmail.com'] # modificada por proteccion de datos

##1.2 EDUNEXT

In [ ]:
edunext_char_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/20260212/characterization/characterization_p1_edunext.csv'


In [ ]:
edunext_char = pd.read_csv(edunext_char_path)

In [ ]:
print("Columnas de archivo edunext de caracterización", edunext_char.columns)

Las columnas para este archivo de caracterización son:

Las columnas para este archivo de caracterización son:

* Unnamed: 0
* username
* ¿Por qué medio te enteraste de este curso?
* ¿En qué nivel consideras que está tu nivel de programación de computadores?
* ¿A qué te dedicas actualmente?
* ¿Cuánto tiempo a la semana planeas dedicar a desarrollar las actividades propuestas en este curso?
* ¿Autoriza al Programa de formación "Machine Learning and Data Science" de la Universidad Nacional de Colombia para usar aparte de sus respuestas en este formulario como ejemplo en futuras ediciones del curso y/o en piezas publicitarias?:
* Si has programado antes, ¿cuál es el lenguaje de programación que más has usado?
* Por favor cuéntanos brevemente ¿Por qué quieres realizar el curso gratuito de Introducción a la programación con Python de la Universidad Nacional de Colombia?
* ¿Cuál es tu principal motivación para aprender a programar con Python?
* email
* name'
* uncode_usr

Se puede ver que son similares a las que tienen los archivos de caracterización de OPENEDX, asi que se procederá a renombrarlas.

In [ ]:
edunext_char = edunext_char.rename(
    columns={
             '¿Por qué medio te enteraste de este curso?': 'dondeconocioelcurso',
              '¿En qué nivel consideras que está tu nivel de programación de computadores?': 'nivelprogramacion',
             '¿A qué te dedicas actualmente?': 'ocupacion',
             '¿Cuánto tiempo a la semana planeas dedicar a desarrollar las actividades propuestas en este curso?': 'dedicacionsemanal',
             'Si has programado antes, ¿cuál es el lenguaje de programación que más has usado?': 'lenguajes_programacion',
             'Por favor cuéntanos brevemente ¿Por qué quieres realizar el curso gratuito de Introducción a la programación con Python de la Universidad Nacional de Colombia?': 'razon_realizar_curso',
             '¿Cuál es tu principal motivación para aprender a programar con Python?': 'motivacion',
             '¿Autoriza al Programa de formación "Machine Learning and Data Science" de la Universidad Nacional de Colombia para usar aparte de sus respuestas en este formulario como ejemplo en futuras ediciones del curso y/o en piezas publicitarias?:': 'autorizacion'

    })

In [ ]:
print("Columnas de archivo edunext de caracterización", edunext_char.columns)

Se eliminan la primera columna de unnamed y la autorización de datos (aunque hay personas que no dieron autorización al uso de sus datos)

In [ ]:
edunext_char["autorizacion"].value_counts()

In [ ]:
edunext_char.drop(columns=['autorizacion'], inplace=True)
edunext_char.drop(columns=['Unnamed: 0'], inplace=True)

In [ ]:
print("Columnas de archivo edunext de caracterización", edunext_char.columns)

Se procede a completar las columnas para unificarlas con los archivos openedx.

In [ ]:
edunext_char.insert(2, 'encuestainicial', 'NA')
edunext_char.insert(8, 'otro_medio', 'NA')
edunext_char.insert(9, 'otra_motivacion', 'NA')
edunext_char.insert(10, 'otro_lenguaje', 'NA')
edunext_char.insert(11, 'otra_ocupacion', 'NA')

print("Columnas de archivo edunext de caracterización", edunext_char.columns)

Se agrega una columna para marcar los registros como Edunext y se verifica que tengan la misma cantidad de columnas y los mismos nombres

In [ ]:
edunext_char['platform'] = 'edunext'

In [ ]:
len(final_openedx_char.columns)

In [ ]:
len(edunext_char.columns)

## Archivo final de caracterización

El archivo final de caracterización cuenta con 1805 registros, tanto de openedx como de edunext.

In [ ]:
final_char = pd.concat([final_openedx_char, edunext_char], ignore_index=True)

In [ ]:
len(final_char)

In [ ]:
final_char.columns

Hay 30 duplicados en el compendio final.
Son personas que han hecho el curso varias veces?

In [ ]:
duplicados_final = final_char[final_char.duplicated(subset=['email'], keep=False)]

In [ ]:
duplicados_ordenados_final = duplicados_final.sort_values(by='email')

In [ ]:
len(duplicados_ordenados_final)

In [ ]:
duplicados_ordenados_final

Se reordenan las columnas y se eliminan unas que no son necesarias

In [ ]:
final_char.columns

Se guarda el archivo de caracterización final en un csv


In [ ]:
final_char.to_csv('/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/import/final-characterization-file.csv', index=False) # index=False prevents writing the index as a separate column

In [ ]:
final_char.columns

# 2.Archivos de autoevaluación

##2.1 OPENEDX

Para openedx se cuenta con 4 archivos diferentes de autoevaluación

In [ ]:
openedx_v1_eval_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/20260212/self_evaluation/self_evaluation_openedx.csv'
openedx_v2_eval_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/20260212/self_evaluation/self_evaluation_openedx_v2.csv'
openedx_p2_v1_eval_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/20260212/self_evaluation/self_evaluation_openedx_p2.csv'
openedx_p2_v2_eval_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/20260212/self_evaluation/self_evaluation_openedx_p2_v2.csv'

In [ ]:
openedx_v1_eval = pd.read_csv(openedx_v1_eval_path)
openedx_v2_eval = pd.read_csv(openedx_v2_eval_path)
openedx_p2_v1_eval = pd.read_csv(openedx_p2_v1_eval_path)
openedx_p2_v2_eval = pd.read_csv(openedx_p2_v2_eval_path)

In [ ]:
print("Columnas de archivo 1 de autoevaluacion", openedx_v1_eval.columns)
print("Columnas de archivo 2 de autoevaluacion", openedx_v2_eval.columns)
print("Columnas de archivo 3 de autoevaluacion", openedx_p2_v1_eval.columns)
print("Columnas de archivo 4 de autoevaluacion", openedx_p2_v2_eval.columns)

In [ ]:
print("Columnas de archivo 1 de autoevaluacion", len(openedx_v1_eval.columns))
print("Columnas de archivo 2 de autoevaluacion", len(openedx_v2_eval.columns))
print("Columnas de archivo 3 de autoevaluacion", len(openedx_p2_v1_eval.columns))
print("Columnas de archivo 4 de autoevaluacion", len(openedx_p2_v2_eval.columns))

### Parte 1

Se evidencia que la v1 y la v2 pueden unirse en un solo dataframe, y ambas p2 también.

Pero antes, se eliminarán columnas que no se necesitan. Del archivo v2 se eliminará la columna de autorización de manejo de los datos.

In [ ]:
openedx_v1_eval.drop(columns=['Unnamed: 0'], inplace=True)

In [ ]:
openedx_v2_eval.drop(columns=['Autoriza'], inplace=True)

In [ ]:
openedx_v2_eval.drop(columns=['Unnamed: 0'], inplace=True)

Para renombrar las columnas cuyos nombres son demasiado extensos y no facilitan el trabajo, se utilizará la siguiente numeración


```
username,username
Autoevaluación,autoevaluacion
Dedicacion semanal,dedicacion_semanal
Foro,foro
Mejora en programacion,pregunta1
Notebooks de Python,pregunta2
Pruebas de código,pregunta3
Retroalimentación,pregunta4
Abstraer un problema en forma de algoritmo e implementarlo en un programa con el lenguaje de programación Python.,pregunta5
"Conocer y aplicar los principios de manejo de variables, tipos de datos y operaciones entre valores numéricos y de texto en Python.",pregunta6
Diseñar e implementar algoritmos con decisiones basadas en la evaluación de expresiones booleanas.,pregunta7
Diseñar e implementar algoritmos con la aplicación de ciclos iterativos.,pregunta8
Distinguir y elegir la colección apropiada para la resolución de un problema con una estructura de datos compleja.,pregunta9
Usar colecciones en el diseño e implementación de algoritmos para la solución de problemas.,pregunta10
Uso de pistas,pregunta11
Utilidad uncode,pregunta12
Videos,pregunta13
Visualización de código,pregunta14
¿Cuál considera que es la mayor fortaleza de este módulo que le ayudó a alcanzar los objetivos de aprendizaje?,pregunta15
¿Qué aspectos considera que podemos mejorar en este curso para que le permita alcanzar en mayor medida los objetivos de aprendizaje?,pregunta16
"Declarar y manipular listas, tuplas, matrices, diccionarios y conjuntos con el lenguaje de programación Python.",pregunta17
email,email
name,name
uncode_usr,uncode_usr
```


In [ ]:
openedx_v1_eval = openedx_v1_eval.rename(columns=
                  {
                    'username' : 'username',
                    'Autoevaluación' : 'autoevaluacion',
                    'Mejora en programacion' : 'pregunta1',
                    'Notebooks de Python' : 'pregunta2',
                    'Pruebas de código' : 'pregunta3',
                    'Retroalimentación' : 'pregunta4',
                    'Abstraer un problema en forma de algoritmo e implementarlo en un programa con el lenguaje de programación Python.' : 'pregunta5',
                    'Conocer y aplicar los principios de manejo de variables, tipos de datos y operaciones entre valores numéricos y de texto en Python.' : 'pregunta6',
                    'Diseñar e implementar algoritmos con decisiones basadas en la evaluación de expresiones booleanas.' : 'pregunta7',
                    'Diseñar e implementar algoritmos con la aplicación de ciclos iterativos.' : 'pregunta8',
                    'Distinguir y elegir la colección apropiada para la resolución de un problema con una estructura de datos compleja.' : 'pregunta9',
                    'Usar colecciones en el diseño e implementación de algoritmos para la solución de problemas.' : 'pregunta10',
                    'Uso de pistas' : 'pregunta11',
                    'Utilidad uncode' : 'pregunta12',
                    'Videos' : 'pregunta13',
                    'Visualización de código' : 'pregunta14',
                    '¿Cuál considera que es la mayor fortaleza de este módulo que le ayudó a alcanzar los objetivos de aprendizaje?' : 'pregunta15',
                    '¿Qué aspectos considera que podemos mejorar en este curso para que le permita alcanzar en mayor medida los objetivos de aprendizaje?' : 'pregunta16',
                    'Declarar y manipular listas, tuplas, matrices, diccionarios y conjuntos con el lenguaje de programación Python.' : 'pregunta17',
                    'Foro' : 'pregunta18',
                    'Dedicacion semanal' : 'pregunta19',
                    'email' : 'email',
                    'name' : 'name',
                    'uncode_usr' : 'uncode_usr'
                    })

In [ ]:
openedx_v2_eval = openedx_v2_eval.rename(columns=
                  {
                    'username' : 'username',
                    'Autoevaluación' : 'autoevaluacion',
                    'Mejora en programacion' : 'pregunta1',
                    'Notebooks de Python' : 'pregunta2',
                    'Pruebas de código' : 'pregunta3',
                    'Retroalimentación' : 'pregunta4',
                    'Abstraer un problema en forma de algoritmo e implementarlo en un programa con el lenguaje de programación Python.' : 'pregunta5',
                    'Conocer y aplicar los principios de manejo de variables, tipos de datos y operaciones entre valores numéricos y de texto en Python.' : 'pregunta6',
                    'Diseñar e implementar algoritmos con decisiones basadas en la evaluación de expresiones booleanas.' : 'pregunta7',
                    'Diseñar e implementar algoritmos con la aplicación de ciclos iterativos.' : 'pregunta8',
                    'Distinguir y elegir la colección apropiada para la resolución de un problema con una estructura de datos compleja.' : 'pregunta9',
                    'Usar colecciones en el diseño e implementación de algoritmos para la solución de problemas.' : 'pregunta10',
                    'Uso de pistas' : 'pregunta11',
                    'Utilidad uncode' : 'pregunta12',
                    'Videos' : 'pregunta13',
                    'Visualización de código' : 'pregunta14',
                    '¿Cuál considera que es la mayor fortaleza de este módulo que le ayudó a alcanzar los objetivos de aprendizaje?' : 'pregunta15',
                    '¿Qué aspectos considera que podemos mejorar en este curso para que le permita alcanzar en mayor medida los objetivos de aprendizaje?' : 'pregunta16',
                    'Declarar y manipular listas, tuplas, matrices, diccionarios y conjuntos con el lenguaje de programación Python.' : 'pregunta17',
                    'Foro' : 'pregunta18',
                    'Dedicacion semanal' : 'pregunta19',
                    'email' : 'email',
                    'name' : 'name',
                    'uncode_usr' : 'uncode_usr'
                    })

In [ ]:
print("Columnas de archivo 1 de autoevaluacion", openedx_v1_eval.columns)
print("Columnas de archivo 2 de autoevaluacion", openedx_v2_eval.columns)

Se limpia el campo email, pues se utilizará como identificador para ver si hay duplicados o si un archivo contiene al otro

In [ ]:
openedx_v1_eval['email'] = openedx_v1_eval['email'].str.strip().str.lower()
openedx_v2_eval['email'] = openedx_v2_eval['email'].str.strip().str.lower()

In [ ]:
len(openedx_v1_eval)

In [ ]:
len(openedx_v2_eval)

In [ ]:
comparacion = openedx_v2_eval[['email']].merge(
    openedx_v1_eval[['email']],
    on='email',
    how='left',
    indicator=True
)

# 1. ¿Está todo contenido? (True/False)
todo_contenido = (comparacion['_merge'] == 'both').all()
emails_faltantes = comparacion[comparacion['_merge'] == 'left_only']['email']

print(f"¿Están todos los emails?: {todo_contenido}")

Se observa que, de 236 emails en el archivo v2, 204 no están contenidos en el archivo v2, por lo que se procede a juntar ambos.

In [ ]:
emails_faltantes

Se concatenan ambos dataframes para tener un compilado de v1 y v2. Se tiene un total de 526 registros al unirlos

In [ ]:
openedx_eval = pd.concat([openedx_v1_eval, openedx_v2_eval], ignore_index=True)

In [ ]:
len(openedx_eval)

Se realiza un análisis de los duplicados por email, hay 64.

In [ ]:
duplicados = openedx_eval[openedx_eval.duplicated(subset=['email'], keep=False)]

# Ordenamos por email para que los repetidos aparezcan juntos
duplicados_ordenados = duplicados.sort_values(by='email')

Se observa que los registros duplicados son filas iguales, por lo que se puede utilizar la funcion drop duplicates y esta elimina las filas iguales.

In [ ]:
duplicados_ordenados

In [ ]:
final_openedx_eval_p1 = openedx_eval.drop_duplicates()

In [ ]:
len(final_openedx_eval_p1)

Aún se observa que hay una fila duplicada, que no tenía los mismos valores. La columna que no permitió la eliminación de estos dos registros fue el valor registrado de usuario de UNCODE, pues en una fila es NAN y en otra sí tiene un valor, por lo que se borra la fila que no brinda información

In [ ]:
duplicados = final_openedx_eval_p1[final_openedx_eval_p1.duplicated(subset=['email'], keep=False)]

# Ordenamos por email para que los repetidos aparezcan juntos
duplicados_ordenados = duplicados.sort_values(by='email')

In [ ]:
duplicados_ordenados

In [ ]:
# Se busca el índice de la fila que cumple las condiciones
indice_a_borrar = final_openedx_eval_p1[(final_openedx_eval_p1['email'] == '@gmail.com') & (final_openedx_eval_p1['uncode_usr'].isna())].index # modificada por proteccion de datos



In [ ]:
indice_a_borrar

In [ ]:
# Borramos por ese índice
final_openedx_eval_p1 = final_openedx_eval_p1.drop(indice_a_borrar)

In [ ]:
len(final_openedx_eval_p1)

Se verifica de forma final que no hayan más duplicados

In [ ]:
duplicados = final_openedx_eval_p1[final_openedx_eval_p1.duplicated(subset=['email'], keep=False)]

# Ordenamos por email para que los repetidos aparezcan juntos
duplicados_ordenados = duplicados.sort_values(by='email')
duplicados_ordenados

### [Deprecated] Parte 2

La parte dos corresponde a la segunda parte del curso, donde no todos participaron debido a que tenía un costo adicional. Por esto, esta parte se descartará. Dejamos el código analizando la parte 2 por si se necesita en algún momento

In [ ]:
openedx_p2_v1_eval = pd.read_csv(openedx_p2_v1_eval_path)
openedx_p2_v2_eval = pd.read_csv(openedx_p2_v2_eval_path)

In [ ]:
print("Columnas de archivo 3 de autoevaluacion", openedx_p2_v1_eval.columns)
print("Columnas de archivo 4 de autoevaluacion", openedx_p2_v2_eval.columns)

In [ ]:
print("Columnas de archivo 3 de autoevaluacion", len(openedx_p2_v1_eval.columns))
print("Columnas de archivo 4 de autoevaluacion", len(openedx_p2_v2_eval.columns))

Se elimina columnas que no se necesitarán

In [ ]:
openedx_p2_v1_eval.drop(columns=['Unnamed: 0'], inplace=True)
openedx_p2_v1_eval.drop(columns=[' Encuesta de Autoevaluación y Retroalimentación (Parte II)'], inplace=True)
openedx_p2_v1_eval.drop(columns=['Autoriza'], inplace=True)

In [ ]:
openedx_p2_v2_eval.drop(columns=['Unnamed: 0'], inplace=True)
openedx_p2_v2_eval.drop(columns=[' Encuesta de Autoevaluación y Retroalimentación (Parte II)'], inplace=True)
openedx_p2_v2_eval.drop(columns=['Autoriza'], inplace=True)

se renombran las columnas. Como esta es la parte 2 del anterior archivo creado, se mapean las columnas con nombre largos que corresponden a preguntan a partir del número 18, pues la parte 1 cuenta con 17 preguntas


pero antes, se verifica si hay preguntas que tal vez estén repetidas entre las dos partes.

In [ ]:
openedx_p2_v1_eval = openedx_p2_v1_eval.rename(columns=
                  {
                      'username' : 'username',
                      'Los recursos dispuestos en forma de notebooks de Python fueron útiles en mi proceso de aprendizaje.' : 'pregunta18',
                      'La posibilidad de hacer pruebas de mis soluciones a los subproblemas de las actividades realizadas dentro de UNCode antes de someter a evaluación las prácticas y tareas fue útil para mi proceso de aprendizaje.' : 'pregunta19',
                      'Los videos de los docentes fueron útiles en mi proceso de aprendizaje.' : 'pregunta20',
                      'Dedicacion semanal' : 'pregunta21',
                      'La sección de Discusión del curso fue útil en mi proceso de aprendizaje.' : 'pregunta22',
                      'Considero que mi conocimiento en el área de programación de computadores ha mejorado gracias a este curso.' : 'pregunta23',
                      'Los recursos interactivos disponibles en la forma de notebooks calificables en UNCode fueron útiles para mi proceso de aprendizaje.' : 'pregunta24',
                      'Las pistas disponibles en las prácticas y tareas dentro de los notebooks calificables fueron útiles para mi proceso de aprendizaje.' : 'pregunta25',
                      'La retroalimentación recibida en la herramienta UNCode fue útil para identificar errores en mis soluciones de prácticas y tareas.' : 'pregunta26',
                      'Comprender y aplicar los conceptos de función, parámetro y variables locales y globales en el desarrollo de programas en Python.' : 'pregunta27',
                      'Conocer los conceptos de objeto, método y atributo y aplicarlos en el diseño e implementación de algoritmos en Python.' : 'pregunta28',
                      'Plantear problemas de análisis de datos y adquirir una visión general de las áreas de aplicación de la ciencia de datos.' : 'pregunta29',
                      'Navegar el sistema de archivos y directorios de un computador e instalar e importar módulos en el lenguaje de programación Python.' : 'pregunta30',
                      'Diseñar e implementar programas básicos de análisis de datos con el paquete de análisis, manipulación y visualización de datos Pandas.' : 'pregunta31',
                      'En general, UNCode fue útil en mi proceso de aprendizaje.' : 'pregunta32',
                      '¿Cuál considera que es la mayor fortaleza de este módulo que le ayudó a alcanzar los objetivos de aprendizaje?' : 'pregunta33',
                      '¿Qué aspectos considera que podemos mejorar en este curso para que le permita alcanzar en mayor medida los objetivos de aprendizaje?' : 'pregunta34',
                      'email' : 'email',
                      'name' : 'name',
                      'uncode_usr' : 'uncode_usr'
                    })

In [ ]:
openedx_p2_v2_eval = openedx_p2_v2_eval.rename(columns=
                  {
                      'username' : 'username',
                      'Los recursos dispuestos en forma de notebooks de Python fueron útiles en mi proceso de aprendizaje.' : 'pregunta18',
                      'La posibilidad de hacer pruebas de mis soluciones a los subproblemas de las actividades realizadas dentro de UNCode antes de someter a evaluación las prácticas y tareas fue útil para mi proceso de aprendizaje.' : 'pregunta19',
                      'Los videos de los docentes fueron útiles en mi proceso de aprendizaje.' : 'pregunta20',
                      'Dedicacion semanal' : 'pregunta21',
                      'La sección de Discusión del curso fue útil en mi proceso de aprendizaje.' : 'pregunta22',
                      'Considero que mi conocimiento en el área de programación de computadores ha mejorado gracias a este curso.' : 'pregunta23',
                      'Los recursos interactivos disponibles en la forma de notebooks calificables en UNCode fueron útiles para mi proceso de aprendizaje.' : 'pregunta24',
                      'Las pistas disponibles en las prácticas y tareas dentro de los notebooks calificables fueron útiles para mi proceso de aprendizaje.' : 'pregunta25',
                      'La retroalimentación recibida en la herramienta UNCode fue útil para identificar errores en mis soluciones de prácticas y tareas.' : 'pregunta26',
                      'Comprender y aplicar los conceptos de función, parámetro y variables locales y globales en el desarrollo de programas en Python.' : 'pregunta27',
                      'Conocer los conceptos de objeto, método y atributo y aplicarlos en el diseño e implementación de algoritmos en Python.' : 'pregunta28',
                      'Plantear problemas de análisis de datos y adquirir una visión general de las áreas de aplicación de la ciencia de datos.' : 'pregunta29',
                      'Navegar el sistema de archivos y directorios de un computador e instalar e importar módulos en el lenguaje de programación Python.' : 'pregunta30',
                      'Diseñar e implementar programas básicos de análisis de datos con el paquete de análisis, manipulación y visualización de datos Pandas.' : 'pregunta31',
                      'En general, UNCode fue útil en mi proceso de aprendizaje.' : 'pregunta32',
                      '¿Cuál considera que es la mayor fortaleza de este módulo que le ayudó a alcanzar los objetivos de aprendizaje?' : 'pregunta33',
                      '¿Qué aspectos considera que podemos mejorar en este curso para que le permita alcanzar en mayor medida los objetivos de aprendizaje?' : 'pregunta34',
                      'email' : 'email',
                      'name' : 'name',
                      'uncode_usr' : 'uncode_usr'
                    })

In [ ]:
print("Columnas de archivo 1 de autoevaluacion", openedx_p2_v1_eval.columns)
print("Columnas de archivo 2 de autoevaluacion", openedx_p2_v2_eval.columns)

Igual que con la parte 1, se realizará un evaluación de duplicados para juntar ambos dataframes en uno solo por medio del email.

In [ ]:
openedx_p2_v1_eval['email'] = openedx_p2_v1_eval['email'].str.strip().str.lower()
openedx_p2_v2_eval['email'] = openedx_p2_v2_eval['email'].str.strip().str.lower()

In [ ]:
len(openedx_p2_v1_eval)

In [ ]:
len(openedx_p2_v2_eval)

NOTA: la parte 2 no parece que fue llenada por muchos estudiantes. Al hacer el análisis de si un archivo contiene a otro, todo el v2 contiene al v1 para la parte 2, por lo que no se tienen que juntar y se puede usar uno solo.

In [ ]:
comparacion = openedx_p2_v2_eval[['email']].merge(
    openedx_p2_v1_eval[['email']],
    on='email',
    how='left',
    indicator=True
)

# 1. ¿Está todo contenido? (True/False)
todo_contenido = (comparacion['_merge'] == 'both').all()
emails_faltantes = comparacion[comparacion['_merge'] == 'left_only']['email']

print(f"¿Están todos los emails?: {todo_contenido}")

In [ ]:
final_openedx_eval_p2 = openedx_p2_v2_eval

Para finalizar la autoevaluación de openedx, se juntará la parte 1 con la parte 2, por medio del email. Por esto, del p2 se eliminan las columnas que ya se encuentran en la p1 (nombre y usuario de UNcode)



In [ ]:
final_openedx_eval_p2.drop(columns=['name'], inplace=True)
final_openedx_eval_p2.drop(columns=['uncode_usr'], inplace=True)
final_openedx_eval_p2.drop(columns=['username'], inplace=True)

In [ ]:
 final_openedx_eval = final_openedx_eval_p1.merge(
    final_openedx_eval_p2,
    on='email',
    how='left',
    indicator=True
)


In [ ]:
print("Columnas de archivo 2 de autoevaluacion", final_openedx_eval.columns)

Al juntar las dos partes para formar un solo datframe con 34 preguntas, solamente 39 personas respondieron a todas las 34 preguntas.

In [ ]:
print(final_openedx_eval['_merge'].value_counts())

### Archivo final openedx

In [ ]:
final_openedx_eval = final_openedx_eval_p1

In [ ]:
final_openedx_eval['platform'] = 'openedx'

In [ ]:
len(final_openedx_eval.columns)

In [ ]:
final_openedx_eval.columns

In [ ]:
columnas_identidad = ['username', 'autoevaluacion', 'email',
                      'name', 'uncode_usr', 'platform']

# 2. Generamos la lista completa de preguntas (del 5 al 36)
# Esto creará: 'pregunta5', 'pregunta6', ..., 'pregunta36'
todas_las_preguntas = [f'pregunta{i}' for i in range(1, 20)]

# 3. Combinamos ambas listas para tener el orden final deseado
orden_final = columnas_identidad + todas_las_preguntas

# 4. Reindexamos el DataFrame
# Esto ordena las existentes y CREA las faltantes como NaN
final_openedx_eval = final_openedx_eval.reindex(columns=orden_final)

In [ ]:
final_openedx_eval.columns

##2.2 EDUNEXT


In [ ]:
edunext_eval_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/20260212/self_evaluation/self_evaluation_p1_edunext.csv'


In [ ]:
edunext_eval = pd.read_csv(edunext_eval_path)

In [ ]:
print("Columnas de archivo 1 de caracterización", edunext_eval.columns)

In [ ]:
len(edunext_eval)

El archivo de autoevaluación de Edunext tiene preguntas similares al archivo final de Openedx, pero no todas son iguales. Se hizo un mappeo de las preguntas para que sean equivalentes a las que ya existen, con el fin de poder crear un solo archivo de autoevaluación al final. Solo hay una pregunta nueva, la 20, respecto a la autorización del uso de los datos

In [ ]:
edunext_eval = edunext_eval.rename(columns=
                  {
                      'username' : 'username',
                      'Distinguir y elegir la colección apropiada para la resolución de un problema con una estructura de datos compleja.' : 'pregunta9',
                      'Los videos de los docentes fueron útiles en mi proceso de aprendizaje.' : 'pregunta13',
                      'Abstraer un problema en forma de algoritmo e implementarlo en un programa con el lenguaje de programación Python.' : 'pregunta5',
                      'Las pistas en las prácticas y tareas dentro de la herramienta UNCode fueron útiles para mi proceso de aprendizaje.' : 'pregunta11',
                      'La retroalimentación recibida en la herramienta UNCode fue útil para identificar errores en mis soluciones de prácticas y tareas.' : 'pregunta4',
                      '¿Qué aspectos considera que podemos mejorar en este curso para que le permita alcanzar en mayor medida los objetivos de aprendizaje?' : 'pregunta16',
                      'Diseñar e implementar algoritmos con decisiones basadas en la evaluación de expresiones booleanas.' : 'pregunta7',
                      'La sección de Discusión del curso fue útil en mi proceso de aprendizaje.' : 'pregunta18',
                      'La herramienta de visualización de la ejecución de código (Python Tutor) dentro de UNCode fue útil para mi proceso de aprendizaje.' : 'pregunta14',
                      'Considero que mi conocimiento en el área de programación de computadores ha mejorado gracias a este curso.' : 'pregunta1',
                      'En general, UNCode fue útil en mi proceso de aprendizaje.' : 'pregunta12',
                      'Usar colecciones en el diseño e implementación de algoritmos para la solución de problemas.' : 'pregunta10',
                      'Conocer y aplicar los principios de manejo de variables, tipos de datos y operaciones entre valores numéricos y de texto en Python.' : 'pregunta6',
                      'Declarar y manipular listas, tuplas, matrices, diccionarios y conjuntos con el lenguaje de programación Python.' : 'pregunta17',
                      'Los recursos dispuestos en forma de notebooks de Python fueron útiles en mi proceso de aprendizaje.' : 'pregunta2',
                      'La posibilidad de hacer pruebas de mis soluciones dentro de UNCode antes de someter a evaluación las prácticas y tareas fue útil para mi proceso de aprendizaje.' : 'pregunta3',
                      'Mi tiempo de trabajo autónomo por semana en actividades relacionadas con este curso (consulta del material audiovisual, estudio de talleres guiados, desarrollo de prácticas y tareas, participación en foros de discusión, otros) fue:' : 'pregunta19',
                      '¿Autoriza al Programa de formación "Machine Learning and Data Science" de la Universidad Nacional de Colombia para usar apartes de sus respuestas en este formulario como ejemplo en futuras ediciones del curso y/o en piezas publicitarias?:' : 'pregunta20',
                      '¿Cuál considera que es la mayor fortaleza de este módulo que le ayudó a alcanzar los objetivos de aprendizaje?' : 'pregunta15',
                      'Diseñar e implementar algoritmos con la aplicación de ciclos iterativos.' : 'pregunta8',
                      'email' : 'email',
                      'name' : 'name',
                      'uncode_usr' : 'uncode_usr'
                    })

In [ ]:
print("Columnas de archivo de caracterización", edunext_eval.columns)

Se elimina la columna de la autorización de los datos

In [ ]:
edunext_eval.drop(columns=['pregunta20'], inplace=True)

### Archivo final edunext

In [ ]:
edunext_eval['platform'] = 'edunext'

In [ ]:
columnas_identidad = ['username', 'autoevaluacion', 'email',
                      'name', 'uncode_usr', 'platform']

todas_las_preguntas = [f'pregunta{i}' for i in range(1, 20)]

orden_final = columnas_identidad + todas_las_preguntas

edunext_eval = edunext_eval.reindex(columns=orden_final)

In [ ]:
edunext_eval.columns

##2.3 Archivo final de autoevaluación

In [ ]:
final_openedx_eval.columns

In [ ]:
final_eval = pd.concat([final_openedx_eval, edunext_eval], ignore_index=True)

Las columnas del archivo final de autoevaluación son:

El equivalente por pregunta está aqui:
https://docs.google.com/spreadsheets/d/1h6ZICzxi7W2_zK9vJ57IwBMI3yPf7CQYFGtBTvAShFI/edit?gid=0#gid=0

In [ ]:
final_eval.columns

In [ ]:
len(final_eval)

In [ ]:
final_eval['platform'].value_counts()

In [ ]:
final_eval.to_csv('/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/import/final-self-evaluation-file.csv', index=False) # index=False prevents writing the index as a separate column

# 3.Archivos de metadata de submissions

### 3.1 Openedx

Cada una de las tareas (submissions) cuenta con un archivo de metadata. Se juntarán todos estos csv en uno solo para Openedx.

In [ ]:
sub_openedx_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/20260212/submissions/open_edx/metadata/' # Replace with your actual path

# Use glob to get a list of all CSV files in the directory
all_sub_openedex_files = glob.glob(os.path.join(sub_openedx_path, "*.csv"))

In [ ]:
all_sub_openedex_files

In [ ]:
df_list = []

# Loop through the file paths, read each file, and append to the list
for filename in all_sub_openedex_files:
    df = pd.read_csv(filename, index_col=None, header=0)
    nombre_con_extension = os.path.basename(filename)
    nombre_actividad = os.path.splitext(nombre_con_extension)[0]
    nombre_actividad = nombre_actividad.replace('_metadata', '')

    df['archivo_origen'] = filename
    df['actividad'] = nombre_actividad
    df_list.append(df)

# Concatenate all DataFrames in the list into a single DataFrame
# ignore_index=True resets the index to be continuous across all combined data
sub_openedex_metadata = pd.concat(df_list, axis=0, ignore_index=True)

# Display the first few rows of the merged DataFrame
print(sub_openedex_metadata.head())
print(f"Total rows in merged DataFrame: {len(sub_openedex_metadata)}")

In [ ]:
len(sub_openedex_metadata)

Estas son las columnas que tiene los submissions de openedx. Hay un total de 64482 submissions.

In [ ]:
sub_openedex_metadata.columns

In [ ]:
sub_openedex_metadata['platform'] = 'openedx'

### 3.2 Edunext

In [ ]:
sub_edunext_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/20260212/submissions/edunext/metadata/' # Replace with your actual path

# Use glob to get a list of all CSV files in the directory
all_sub_edunext_files = glob.glob(os.path.join(sub_edunext_path, "*.csv"))

In [ ]:
df_list = []

# Loop through the file paths, read each file, and append to the list
for filename in all_sub_edunext_files:
    df = pd.read_csv(filename, index_col=None, header=0)
    nombre_con_extension = os.path.basename(filename)
    nombre_actividad = os.path.splitext(nombre_con_extension)[0]
    nombre_actividad = nombre_actividad.replace('_metadata', '')

    df['archivo_origen'] = filename
    df['actividad'] = nombre_actividad
    df_list.append(df)

# Concatenate all DataFrames in the list into a single DataFrame
# ignore_index=True resets the index to be continuous across all combined data
sub_edunext_metadata = pd.concat(df_list, axis=0, ignore_index=True)

# Display the first few rows of the merged DataFrame
print(sub_edunext_metadata.head())
print(f"Total rows in merged DataFrame: {len(sub_edunext_metadata)}")

Hay un total de 112254 submission provenientes de Edunext

In [ ]:
len(sub_edunext_metadata)

In [ ]:
sub_edunext_metadata.columns

In [ ]:
sub_edunext_metadata['platform'] = 'edunext'

### 3.3 Archivo final de metada de submissions


Se puede ver que las columnas de la metadata proveniente de Edunext y Openedx son las mismas, por lo que se pueden concatenar para formar un solo archivo de metadata de submissions

In [ ]:
final_submission = pd.concat([sub_openedex_metadata, sub_edunext_metadata], ignore_index=True)

In [ ]:
len(final_submission)

In [ ]:
final_submission.columns

In [ ]:
final_submission.to_csv('/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/import/final-submissions-file.csv', index=False) # index=False prevents writing the index as a separate column